# 09. Alert-capacity design under clustered crashes

In the spot results, 33 of the 44 events that the calendar policy (20 per month) missed, 40% of all 83 events, occurred after that month's quota was already used up, and crashes cluster (Hawkes branching ratio ≈ 0.58, mean aftershock lifetime ≈ 9 days, estimated on 2021–22). The risk-adaptive bucket of notebook 06 used less of its budget because its mean weight was below 1.

I compare four policies, all with a hard cap of 240 alerts per year, a 240-minute cooldown and the same V+F score:
- P0 calendar: 20 per month (the notebook 05 baseline).
- P1 constant bucket: burst 20, no monthly cap, at the fixed notebook 06 threshold.
- P2 budget-neutral risk bucket: the notebook 06 risk weights divided by their mean over the preceding 365 days so that the mean is 1 (calibrated on 2022).
- P3 surge-reserve calendar: a normal quota of 16 per month (80% of 20) plus a reserve that accrues 4 per month (48 per year, reset at the start of each year). The policy can draw on the reserve only in a "cluster state", in which a 4% crash was *confirmed* within the previous 9 days, and only after it has used the month's 16 alerts. Cumulative alerts therefore never exceed the baseline pace (20 per month) at the end of any month, and the annual total stays the same, 16×12 + 48 = 240. The crash confirmation time (`confirmed_utc`) is price information actually available at that moment, so the rule is causal. The 9-day window is the Hawkes estimate from notebook 04 (2021–22), rounded up.

I use Binance spot BTCUSDT one-minute bars, evaluate on 2023-01-01 to 2026-04-30 and write the results to `btc_lead_cost_results_77months_spot/capacity_design_v1/`. This analysis is exploratory. I write the candidate-selection rule into the specification `capacity_spec.json` before computing results.

In [2]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
# --- emit_policy, as in notebook 06 ---
def emit_policy_nb6(t, scores, threshold, mode, *, credits=None, B=240, month_cap=20,
                burst=20, cooldown=240):
    t=np.asarray(t,np.int64);sc=np.asarray(scores,float)
    assert len(t)==len(sc) and (len(t)<2 or (np.diff(t)>0).all())
    if mode=='risk_bucket':
        assert credits is not None and len(credits)==len(t)
        credits=np.asarray(credits,float)
    assert mode in {'calendar','constant_bucket','risk_bucket'}
    cand=np.flatnonzero(np.isfinite(sc)&(sc>=threshold))
    emitted=[];annual=0;monthly=0;py=None;pm=None;prev=-10**25
    tokens=float(burst); last_t=None;last_credit=None
    for k in cand:
        tm=int(t[k]);stamp=pd.Timestamp(tm,tz='UTC')
        yr=int(stamp.year);mon=int(stamp.month)
        if yr!=py:
            py=yr;annual=0;tokens=float(burst);last_t=tm
            last_credit=None if credits is None else float(credits[k])
        if (yr,mon)!=pm:
            pm=(yr,mon);monthly=0
        if mode=='constant_bucket':
            if last_t is not None and tm>last_t:
                tokens=min(float(burst),tokens+(tm-last_t)/NS*B/(365.2425*1440))
            last_t=tm
        elif mode=='risk_bucket':
            if last_credit is not None:
                tokens=min(float(burst),tokens+max(0,float(credits[k])-last_credit))
            last_credit=float(credits[k])
        if annual>=B:continue
        if mode=='calendar' and monthly>=month_cap:continue
        if mode!='calendar' and tokens<1-1e-9:continue
        if tm-prev<cooldown*NS:continue
        emitted.append(tm);prev=tm;annual+=1;monthly+=1
        if mode!='calendar':tokens=max(0.0,tokens-1.0)
    return np.array(emitted,dtype=np.int64)


RISK_MIN = 0.5; RISK_MAX = 2.0; TRAILING_DAYS = 7


def daily_risk_weights(idx, close, volume, ref=None):
    """Notebook 06 causal daily risk weight (trailing 7 complete UTC days of 1m squared returns / 2021 median),
    plus the budget-neutral variant: divided by its own trailing 365-day mean (previous days only), re-clipped."""
    logclose = np.log(close)
    ret = np.r_[np.nan, np.diff(logclose)]
    usable = (np.isfinite(ret) & np.isfinite(close) & (close > 0) & (volume > 0) &
              np.r_[False, (volume[:-1] > 0) & np.isfinite(close[:-1]) & (close[:-1] > 0)])
    ret2 = pd.Series(np.where(usable, ret * ret, np.nan), index=idx)
    D = pd.DataFrame({'ms': ret2.resample('D').mean(), 'n': ret2.resample('D').count()})
    D.loc[D.n < 1000, 'ms'] = np.nan
    D['trailing_7d_rv'] = np.sqrt(D.ms.shift(1).rolling(TRAILING_DAYS, min_periods=TRAILING_DAYS).mean())
    if ref is None:
        ref = D.loc['2021-01-01':'2021-12-31', 'trailing_7d_rv'].median()
    assert np.isfinite(ref) and ref > 0
    D['risk_weight'] = np.clip(D.trailing_7d_rv / ref, RISK_MIN, RISK_MAX).fillna(1.0)
    norm = D.risk_weight.shift(1).rolling(365, min_periods=90).mean()
    D['risk_weight_bn'] = np.clip(D.risk_weight / norm, RISK_MIN, RISK_MAX).fillna(1.0)
    return D, float(ref)


def credits(idx, daily_weight):
    w = daily_weight.reindex(idx.floor('D')).to_numpy(float)
    assert np.isfinite(w).all()
    return np.cumsum(B / (365.2425 * 1440) * w, dtype=np.float64)


def calibrate_bucket(t_cal, s_cal, mode, credits_cal=None, grid=None):
    """Notebook 06 rule: closest to 240 alerts in 2022, strictest threshold on ties."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        a = emit_policy_nb6(t_cal, s_cal, float(th), mode, credits=credits_cal if mode == 'risk_bucket' else None)
        rows.append((float(th), len(a), abs(len(a) - B)))
    df = pd.DataFrame(rows, columns=['threshold', 'calibration_alarms', 'distance_to_240'])
    row = df.sort_values(['distance_to_240', 'threshold'], ascending=[True, False]).iloc[0]
    return float(row.threshold), int(row.calibration_alarms), df
SURGE_BASE_CAP = 16                        # regular alerts per month (80% of 20)
SURGE_RESERVE = B - 12 * SURGE_BASE_CAP    # 48 alerts per UTC year usable only during a crash cluster
SURGE_ACCRUAL = SURGE_RESERVE // 12        # the reserve accrues 4 per month, so by the end of month m at most 20*m alerts


def make_surge_flags(confirmed_ns, surge_days):
    conf = np.sort(np.asarray(confirmed_ns, np.int64))

    def flags(t):
        j = np.searchsorted(conf, t, 'left') - 1          # last crash confirmed strictly before minute t
        out = np.zeros(len(t), bool); ok = j >= 0
        out[ok] = (t[ok] - conf[j[ok]]) < surge_days * 1440 * NS
        return out
    return flags


@njit(cache=False)
def _reserve_core(t, elig, surge, years, months, B_, base_cap, reserve, accrual, cooldown_ns):
    out = np.empty(len(t), np.int64); kind = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; rused = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; rused = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B_ or ti - last < cooldown_ns:
            continue
        if mcount < base_cap:
            mcount += 1; kind[k] = 0
        elif surge[i] and rused < min(reserve, accrual * (mo - yr * 12)):
            rused += 1; kind[k] = 1
        else:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1
    return out[:k].copy(), kind[:k].copy()


def emit_reserve_calendar(t, eligible, surge, base_cap=SURGE_BASE_CAP, reserve=SURGE_RESERVE, B_=240, cooldown_min=240):
    """Calendar policy with a base monthly quota plus a reserve that accrues reserve/12 per month within the UTC year
    and can only be spent after the month quota is used and while a crash was confirmed in the previous surge_days.
    Cumulative alerts never exceed the 20/month calendar pace. Returns (alert times, kinds 0=base 1=reserve)."""
    t = np.asarray(t, np.int64)
    assert len(t) < 2 or np.all(np.diff(t) > 0)
    years, months = cal_fields(t)
    return _reserve_core(t, np.asarray(eligible, bool), np.asarray(surge, bool), years, months, int(B_), int(base_cap),
                         int(reserve), int(reserve // 12), np.int64(cooldown_min * NS))


def emit_reserve_calendar_py(t, eligible, surge, base_cap=SURGE_BASE_CAP, reserve=SURGE_RESERVE, B_=240, cooldown_min=240):
    """Plain-Python reference used only to check the compiled version."""
    out = []; kinds = []; mcount = ycount = rused = 0; pm = py = None; last = -(10**20)
    for i in np.flatnonzero(eligible):
        ti = int(t[i]); d = pd.Timestamp(ti, tz='UTC'); yr = d.year; mo = yr * 12 + d.month
        if yr != py:
            ycount = 0; rused = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B_ or ti - last < cooldown_min * NS:
            continue
        if mcount < base_cap:
            mcount += 1; kinds.append(0)
        elif surge[i] and rused < min(reserve, (reserve // 12) * d.month):
            rused += 1; kinds.append(1)
        else:
            continue
        out.append(ti); last = ti; ycount += 1
    return np.asarray(out, np.int64), np.asarray(kinds, np.int64)


def check_reserve(t, s, th, surge):
    m = in_period(t, '2022-06-01', '2023-01-01') if (t >= ts('2022-06-01').value).any() else np.ones(len(t), bool)
    a, ka = emit_reserve_calendar(t[m], s[m] >= th, surge[m]); b, kb = emit_reserve_calendar_py(t[m], s[m] >= th, surge[m])
    if not (np.array_equal(a, b) and np.array_equal(ka, kb)):
        raise AssertionError('compiled reserve calendar differs from the Python reference')
    return len(a)

OUT = SPOT_BASE / 'capacity_design_v1'
OUT.mkdir(parents=True, exist_ok=True)
HAWKES = SPOT_BASE / 'systems_engineering_scheduler_oracle_value_v1' / 'EXPLORATORY_hawkes_development_only.csv'
if not HAWKES.is_file():
    raise FileNotFoundError(f'Results of notebook 04 are required: {HAWKES}')
hk = pd.read_csv(HAWKES).iloc[0]
SURGE_DAYS = int(math.ceil(float(hk.mean_excitation_life_days)))
SPEC = {'study': 'alert capacity design, exploratory', 'annual_hard_cap': B, 'cooldown_minutes': COOLDOWN,
        'score': 'V+F logistic, 2021 train (identical to notebook 5)',
        'P0': 'calendar 20/month, TH_CAL', 'P1': 'constant bucket burst 20 (notebook 6), TH_BUCKET',
        'P2': 'risk bucket with weights / trailing-365-day mean weight (previous days only), clipped [0.5, 2]; threshold: 2022 alert count closest to 240, strictest on ties',
        'P3': f'calendar with base quota {SURGE_BASE_CAP}/month plus a reserve accruing {SURGE_ACCRUAL}/month (max {SURGE_RESERVE}/UTC year, reset each year) usable only after the month quota is used AND while a 4% crash was confirmed within the previous {SURGE_DAYS} days; annual cap {B}; TH_CAL',
        'surge_days_source': f'ceil(mean_excitation_life_days={float(hk.mean_excitation_life_days):.3f}) from notebook-4 Hawkes fit on 2021-22',
        'evaluation': f'{EVAL_START} to {EVAL_END} exclusive', 'primary_endpoint': 'pre-1%-onset hits (30-240 min before breach)',
        'candidate_rule': 'among P2 and P3: most pre-onset hits; passed to notebook 11 only if strictly more pre-onset hits than P0 '
                          'AND realized alerts <= 1.02 x P0 alerts; ties -> P3 (no recalibration needed)',
        'status': 'exploratory; 2023-2026 previously inspected'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True) + '\n').encode()
(OUT / 'capacity_spec.json').write_bytes(_raw)
(OUT / 'capacity_spec.sha256').write_text(hashlib.sha256(_raw).hexdigest() + '  capacity_spec.json\n')
print('Output:', OUT, '| surge days', SURGE_DAYS, '| base quota', SURGE_BASE_CAP, '| annual reserve', SURGE_RESERVE)

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/capacity_design_v1 | surge days 9 | base quota 16 | annual reserve 48


## 1. Score and baseline reproduction

In [3]:
px, sg = load_btc_spot()
E = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
idx = px.index; T = idx.asi8
close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v = sg.z_vol.to_numpy(float); f = sg.tsr.to_numpy(float); h = sg.z_hs.to_numpy(float)
valid = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) &
         np.isfinite(v) & np.isfinite(f) & np.isfinite(h) & (f >= 0) & (f <= 1))
y = label_from_catalog(T, E)
X = np.column_stack([v, f])
tr = train_mask(T, valid)
clf = new_logit().fit(X[tr], y[tr])
S = np.full(len(T), np.nan); S[valid] = clf.predict_proba(X[valid])[:, 1]
cal = valid & in_period(T, '2022-01-01', '2023-01-01'); ev = valid & in_period(T, EVAL_START, EVAL_END)
EVENTS = events_between(E, EVAL_START, EVAL_END)
assert len(EVENTS) == REF['n_primary_events']
T_CAL, S_CAL, T_EV, S_EV = T[cal], S[cal], T[ev], S[ev]
print('fast calendar identical to the reference on', check_fast_calendar(T_CAL, S_CAL), 'checks')
th0, _, _ = calibrate_calendar(T_CAL, S_CAL)
if abs(th0 - TH_CAL) > 1e-12:
    raise RuntimeError(f'threshold {th0} != notebook 5 {TH_CAL}')
A = {}
A['P0_calendar'] = emit_calendar_fast(T_EV, S_EV >= TH_CAL)
s0 = statistics(EVENTS, A['P0_calendar'])
if [s0['alarms'], s0['caught'], s0['before_1pct']] != REF['first_eligible']:
    raise RuntimeError(f'P0 {s0} != notebook 5 {REF["first_eligible"]}')
print('PASS: P0 reproduces notebook 5:', REF['first_eligible'])

fast calendar identical to the reference on 14 checks
PASS: P0 reproduces notebook 5: [800, 39, 22]


## 2. Reproduce P1, calibrate P2, build P3

In [4]:
D, ref = daily_risk_weights(idx, close, volume)
nb6_days = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'causal_risk_weight_by_utc_day.csv'
if nb6_days.is_file():
    old = pd.read_csv(nb6_days, index_col=0)
    w_old = old['risk_weight'].to_numpy(float); w_new = D['risk_weight'].to_numpy(float)
    assert len(w_old) == len(w_new) and np.allclose(w_old, w_new, rtol=0, atol=1e-12), 'risk weights differ from notebook 6'
    print('PASS: notebook-6 daily risk weights reproduced')
C_STD = credits(idx, D.risk_weight); C_BN = credits(idx, D.risk_weight_bn)
print('mean daily weight 2022 / 2023-26 — notebook 6:', round(D.loc['2022', 'risk_weight'].mean(), 3), round(D.loc['2023':'2026-04', 'risk_weight'].mean(), 3),
      '| budget-neutral:', round(D.loc['2022', 'risk_weight_bn'].mean(), 3), round(D.loc['2023':'2026-04', 'risk_weight_bn'].mean(), 3))

A['P1_constant_bucket'] = emit_policy_nb6(T_EV, S_EV, TH_BUCKET, 'constant_bucket')
nb6_tab = pd.read_csv(SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'risk_adaptive_policy_comparison.csv').set_index('policy')
r1 = nb6_tab.loc['constant_bucket_recalibrated']
s1 = statistics(EVENTS, A['P1_constant_bucket'])
if [s1['alarms'], s1['caught'], s1['before_1pct']] != [int(r1.alarms), int(r1.caught), int(r1.pre_onset)]:
    raise RuntimeError(f'P1 {s1} != notebook 6 constant bucket {r1.to_dict()}')
print('PASS: P1 reproduces notebook 6 constant bucket:', [int(r1.alarms), int(r1.caught), int(r1.pre_onset)])

TH_BN, n_bn, grid_bn = calibrate_bucket(T_CAL, S_CAL, 'risk_bucket', credits_cal=C_BN[cal])
grid_bn.to_csv(OUT / 'P2_threshold_search_2022.csv', index=False)
A['P2_risk_bucket_budget_neutral'] = emit_policy_nb6(T_EV, S_EV, TH_BN, 'risk_bucket', credits=C_BN[ev])
print('P2 threshold (2022):', TH_BN, '2022 alerts:', n_bn)

surge_flags = make_surge_flags(E.confirmed_utc.astype('int64').to_numpy(), SURGE_DAYS)
print('reserve calendar identical to the Python reference on', check_reserve(T_CAL, S_CAL, TH_CAL, surge_flags(T_CAL)), 'alerts')
A['P3_surge_reserve_calendar'], KIND3 = emit_reserve_calendar(T_EV, S_EV >= TH_CAL, surge_flags(T_EV))
n_p3_2022 = len(emit_reserve_calendar(T_CAL, S_CAL >= TH_CAL, surge_flags(T_CAL))[0])
print('P3 alerts in 2022 (same threshold, informational):', n_p3_2022, '| 2023-26 reserve alerts used:', int(KIND3.sum()),
      '| share of 2023-26 minutes in surge state:', round(float(surge_flags(T_EV).mean()), 3))

PASS: notebook-6 daily risk weights reproduced
mean daily weight 2022 / 2023-26 — notebook 6: 0.796 0.614 | budget-neutral: 0.873 0.972
PASS: P1 reproduces notebook 6 constant bucket: [795, 48, 18]
P2 threshold (2022): 0.6698990591069133 2022 alerts: 229
reserve calendar identical to the Python reference on 156 alerts
P3 alerts in 2022 (same threshold, informational): 236 | 2023-26 reserve alerts used: 152 | share of 2023-26 minutes in surge state: 0.389


## 3. Results: hits, paired comparisons, monthly usage, capacity decomposition

In [5]:
rows = []
for k, a in A.items():
    st = statistics(EVENTS, a)
    rows.append(dict(policy=k, **st))
    pd.DataFrame({'alarm_utc': pd.to_datetime(a, utc=True)}).to_csv(OUT / f'alarms_{k}.csv', index=False)
    match_events(EVENTS, a).to_csv(OUT / f'matched_{k}.csv', index=False)
res = pd.DataFrame(rows); res.to_csv(OUT / 'capacity_policy_comparison.csv', index=False)
display(res.round(4))
pairs = pd.DataFrame([dict(policy=k, **paired(EVENTS, A['P0_calendar'], a)) for k, a in A.items() if k != 'P0_calendar'])
pairs.to_csv(OUT / 'paired_vs_P0.csv', index=False)
print('Paired against P0 (exact McNemar; exploratory):'); display(pairs.round(4))

months = pd.period_range('2023-01', '2026-04', freq='M')
mon = pd.DataFrame({k: pd.Series(1, index=pd.to_datetime(a, utc=True).tz_localize(None).to_period('M')).groupby(level=0).sum()
                    .reindex(months, fill_value=0) for k, a in A.items()})
mon.to_csv(OUT / 'monthly_alerts_by_policy.csv')
print('Monthly realized alerts (first rows):'); display(mon.head(12))


def room_decomposition(events, alerts, room_fn):
    """For each event: caught / missed with no room at any minute of its warning window / missed with some room."""
    hit = set(match_events(events, alerts).event_id)
    out = []
    for e in events.itertuples(index=False):
        b = int(e.breach_utc.value)
        tt = np.arange(b - LEAD_MAX * NS, b - LEAD_MIN * NS + 1, NS, dtype=np.int64)
        room = bool(room_fn(tt).any())
        cat = 'caught' if int(e.event_id) in hit else ('missed_no_room' if not room else 'missed_some_room')
        out.append(dict(event_id=int(e.event_id), breach_utc=e.breach_utc, category=cat))
    return pd.DataFrame(out)


def calendar_room(alerts, cap):
    a = np.sort(np.asarray(alerts, np.int64)); am = cal_fields(a)[1] if len(a) else np.empty(0, np.int64)

    def room(tt):
        _, mt = cal_fields(tt); n_before = np.searchsorted(a, tt, 'right')
        return np.array([int((am[:n] == m).sum()) < cap for n, m in zip(n_before, mt)])
    return room


def reserve_room(alerts, kinds):
    a = np.asarray(alerts, np.int64); kinds = np.asarray(kinds); yrs, am = cal_fields(a) if len(a) else (np.empty(0, np.int64),) * 2

    def room(tt):
        yt, mt = cal_fields(tt); n_before = np.searchsorted(a, tt, 'right'); fl = surge_flags(tt); res = []
        for n, y_, m, f_ in zip(n_before, yt, mt, fl):
            base_used = int(((am[:n] == m) & (kinds[:n] == 0)).sum()); res_used = int(((yrs[:n] == y_) & (kinds[:n] == 1)).sum())
            year_used = int((yrs[:n] == y_).sum())
            res.append(year_used < B and (base_used < SURGE_BASE_CAP or (f_ and res_used < min(SURGE_RESERVE, SURGE_ACCRUAL * (m - y_ * 12)))))
        return np.array(res)
    return room


dec0 = room_decomposition(EVENTS, A['P0_calendar'], calendar_room(A['P0_calendar'], MONTH_CAP))
dec3 = room_decomposition(EVENTS, A['P3_surge_reserve_calendar'], reserve_room(A['P3_surge_reserve_calendar'], KIND3))
d0 = dec0.category.value_counts().to_dict(); d3 = dec3.category.value_counts().to_dict()
print('P0 capacity decomposition:', d0); print('P3 capacity decomposition:', d3)
nb6_dec = pd.read_csv(SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'baseline_capacity_event_decomposition.csv').category.value_counts()
assert d0.get('caught', 0) == int(nb6_dec.get('caught', 0)) and d0.get('missed_no_room', 0) == int(nb6_dec.get('missed_no_calendar_room', 0)), \
    'P0 decomposition differs from notebook 6'
print('PASS: P0 decomposition matches notebook 6')
pd.concat([dec0.assign(policy='P0_calendar'), dec3.assign(policy='P3_surge_reserve_calendar')]).to_csv(OUT / 'capacity_decomposition.csv', index=False)

yrs = []
for k, a in A.items():
    for year in range(2023, 2027):
        ev_y = EVENTS[EVENTS.breach_utc.dt.year == year]; a_y = a[pd.to_datetime(a, utc=True).year == year]
        s_ = statistics(ev_y, a_y) if len(ev_y) else dict(caught=0, before_1pct=0)
        yrs.append(dict(policy=k, year=year, alarms=len(a_y), events=len(ev_y), caught=s_['caught'], before_1pct=s_['before_1pct']))
yrs = pd.DataFrame(yrs); yrs.to_csv(OUT / 'capacity_year_by_year.csv', index=False)
display(yrs.pivot(index='year', columns='policy', values='before_1pct'))

,policy,alarms,caught,before_1pct,false_alarms,recall,pre_onset_recall,alert_precision
0,P0_calendar,800,39,22,761,0.4699,0.2651,0.0488
1,P1_constant_bucket,795,48,18,747,0.5783,0.2169,0.0604
2,P2_risk_bucket_budget_neutral,808,35,14,773,0.4217,0.1687,0.0433
3,P3_surge_reserve_calendar,792,36,21,756,0.4337,0.2530,0.0455


Paired against P0 (exact McNemar; exploratory):


,policy,base_caught,new_caught,base_only_caught,new_only_caught,p_caught_two_sided,base_pre1pct,new_pre1pct,base_only_pre1pct,new_only_pre1pct,p_pre1pct_two_sided,p_pre1pct_one_sided_new_better
0,P1_constant_bucket,39,48,10,19,0.1360,22,18,11,7,0.4807,0.8811
1,P2_risk_bucket_budget_neutral,39,35,17,13,0.5847,22,14,12,4,0.0768,0.9894
2,P3_surge_reserve_calendar,39,36,4,1,0.3750,22,21,2,1,1.0000,0.8750


Monthly realized alerts (first rows):


,P0_calendar,P1_constant_bucket,P2_risk_bucket_budget_neutral,P3_surge_reserve_calendar
2023-01,20,36,35,20
2023-02,20,16,13,20
2023-03,20,18,20,20
2023-04,20,19,15,20
2023-05,20,19,16,20
2023-06,20,26,15,20
2023-07,20,15,16,19
2023-08,20,24,19,21
2023-09,20,18,16,16
2023-10,20,21,19,24


P0 capacity decomposition: {'caught': 39, 'missed_no_room': 33, 'missed_some_room': 11}
P3 capacity decomposition: {'missed_no_room': 37, 'caught': 36, 'missed_some_room': 10}
PASS: P0 decomposition matches notebook 6


policy,P0_calendar,P1_constant_bucket,P2_risk_bucket_budget_neutral,P3_surge_reserve_calendar
year,,,,
2023,3,3,2,3
2024,7,7,4,7
2025,9,6,6,8
2026,3,2,2,3


## 4. Candidate selection by the pre-specified rule (passed to notebook 11)

In [6]:
r = res.set_index('policy')
base_pre = int(r.loc['P0_calendar', 'before_1pct']); base_alarms = int(r.loc['P0_calendar', 'alarms'])
order = ['P3_surge_reserve_calendar', 'P2_risk_bucket_budget_neutral']           # tie-break: P3 first
cands = sorted(order, key=lambda k: (-int(r.loc[k, 'before_1pct']), order.index(k)))
best = cands[0]
ok = int(r.loc[best, 'before_1pct']) > base_pre and int(r.loc[best, 'alarms']) <= 1.02 * base_alarms
selected = best if ok else None
cand = {'selected': selected, 'rule': SPEC['candidate_rule'],
        'parameters': {'P3_surge_reserve_calendar': {'threshold': TH_CAL, 'surge_days': SURGE_DAYS, 'base_cap': SURGE_BASE_CAP, 'reserve': SURGE_RESERVE},
                       'P2_risk_bucket_budget_neutral': {'threshold': TH_BN, 'burst': 20, 'weights': 'risk_weight_bn',
                                                         'risk_ref_2021': ref}}.get(selected) if selected else None,
        'table': {k: {'alarms': int(r.loc[k, 'alarms']), 'caught': int(r.loc[k, 'caught']), 'before_1pct': int(r.loc[k, 'before_1pct'])} for k in r.index},
        'spec_sha256': hashlib.sha256(_raw).hexdigest()}
(OUT / 'capacity_candidate.json').write_text(json.dumps(cand, indent=2, default=float))
print(json.dumps(cand, indent=2, default=float))

{
  "selected": null,
  "rule": "among P2 and P3: most pre-onset hits; passed to notebook 11 only if strictly more pre-onset hits than P0 AND realized alerts <= 1.02 x P0 alerts; ties -> P3 (no recalibration needed)",
  "parameters": null,
  "table": {
    "P0_calendar": {
      "alarms": 800,
      "caught": 39,
      "before_1pct": 22
    },
    "P1_constant_bucket": {
      "alarms": 795,
      "caught": 48,
      "before_1pct": 18
    },
    "P2_risk_bucket_budget_neutral": {
      "alarms": 808,
      "caught": 35,
      "before_1pct": 14
    },
    "P3_surge_reserve_calendar": {
      "alarms": 792,
      "caught": 36,
      "before_1pct": 21
    }
  },
  "spec_sha256": "a7f0ea2bcd71d0d65c677fd042d625e73cb4cf093394de5472d4a42faf4889c5"
}
